In [ ]:
!pip install sentence-transformers pdfplumber

import pdfplumber
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 55.5 MB/s eta 0:00:00


In [ ]:
uploaded = files.upload()

Saving resume-sample(5).pdf to resume-sample(5).pdf
Saving resume-sample(4).pdf to resume-sample(4).pdf
Saving resume-sample(3).pdf to resume-sample(3).pdf
Saving resume-sample(2).pdf to resume-sample(2).pdf
Saving resume-sample(1).pdf to resume-sample(1).pdf


In [ ]:
!pip install reportlab

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

def create_pdf(filename, text):
    doc = SimpleDocTemplate(filename)
    styles = getSampleStyleSheet()

    content = []
    for line in text.split("\n"):
        content.append(Paragraph(line, styles["Normal"]))

    doc.build(content)

# Resume 1 (Perfect Match)
resume1 = """John Doe

Professional Summary:
Experienced professional with 5+ years of experience in management, communication, and data handling.

Skills:
- Management
- Communication
- Excel
- Reporting

Experience:
Worked as a team leader managing operations and handling organizational tasks efficiently for over 5 years.

Education:
B.Tech in Computer Science"""
create_pdf("perfect_resume.pdf", resume1)

# Resume 2 (Irrelevant)
resume2 = """Jane Smith

Professional Summary:
Creative individual with passion for cooking, painting, and music.

Skills:
- Cooking
- Dancing
- Painting

Experience:
Worked as a freelance artist and chef.

Education:
Diploma in Fine Arts"""
create_pdf("irrelevant_resume.pdf", resume2)

In [ ]:
from google.colab import files
files.download("perfect_resume.pdf")
files.download("irrelevant_resume.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving irrelevant_resume.pdf to irrelevant_resume.pdf
Saving perfect_resume.pdf to perfect_resume.pdf
Saving resume-sample(1).pdf to resume-sample(1).pdf
Saving resume-sample(2).pdf to resume-sample(2).pdf
Saving resume-sample(3).pdf to resume-sample(3).pdf
Saving resume-sample(4).pdf to resume-sample(4).pdf
Saving resume-sample(5).pdf to resume-sample(5).pdf


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def extract_text_from_pdf(file_name):
    text = ""
    with pdfplumber.open(file_name) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s.]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [ ]:
def split_sentences(text):
    sentences = re.split(r'\.', text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

In [ ]:
skills_dict = {
    "python": ["python"],
    "machine learning": ["ml", "machine learning", "pytorch"],
    "frontend": ["react", "frontend", "javascript"],
    "database": ["sql", "mysql", "database"],
    "accounting": ["accounting", "finance", "bookkeeping"],
    "excel": ["excel"],
    "communication": ["communication"],
    "management": ["management"]
}

def extract_skills(text):
    text = text.lower()
    found_skills = []

    for skill, keywords in skills_dict.items():
        for word in keywords:
            if word in text:
                found_skills.append(skill)
                break

    return found_skills

In [ ]:
def experience_score(text):
    matches = re.findall(r'(\d+)\s*(?:\+)?\s*years', text.lower())

    if matches:
        years = max([int(x) for x in matches])
        return min(years / 10, 1)

    if "intern" in text.lower():
        return 0.3
    elif "project" in text.lower():
        return 0.2

    return 0.1

In [ ]:
def confidence_level(score):
    if score > 70:
        return "High"
    elif score > 50:
        return "Medium"
    else:
        return "Low"

In [ ]:
def generate_justification(candidate):
    skills = candidate['skills']
    semantic = candidate['semantic']

    if candidate['score'] > 60:
        return f"Strong candidate with good semantic alignment ({semantic:.2f}%) and relevant skills {skills[:3]}."
    elif candidate['skill_score'] > 50:
        return f"Candidate has relevant skills {skills[:2]} but moderate semantic alignment ({semantic:.2f}%)."
    elif semantic > 45:
        return "Good contextual understanding but lacks key required skills."
    else:
        return "Low match due to weak alignment in both skills and experience."

In [ ]:
job_description = """
Looking for a professional with experience in management, communication,
data handling, reporting, and organizational skills.
"""

In [ ]:
results = []

jd_clean = clean_text(job_description)
jd_embedding = model.encode(jd_clean)

jd_skills = extract_skills(job_description)

for file_name in uploaded.keys():

    resume_text = extract_text_from_pdf(file_name)
    resume_clean = clean_text(resume_text)

    sentences = split_sentences(resume_clean)

    similarities = []

    for sentence in sentences:
        sent_embedding = model.encode(sentence)
        sim = cosine_similarity([sent_embedding], [jd_embedding])[0][0]
        similarities.append(sim)

    # 🔥 TOP-K SEMANTIC (UPGRADE)
    top_k = sorted(similarities, reverse=True)[:3]
    semantic = sum(top_k) / len(top_k) if top_k else 0

    skills = extract_skills(resume_text)

    # 🔥 IMPROVED SKILL SCORE
    skill_match = len(set(skills) & set(jd_skills))
    skill = skill_match / len(jd_skills) if jd_skills else 0

    experience = experience_score(resume_text)

    final_score = (
        0.5 * semantic +
        0.3 * skill +
        0.2 * experience
    )


    missing_skills = list(set(jd_skills) - set(skills))

    results.append({
        "name": file_name,
        "score": final_score * 100,
        "semantic": semantic * 100,
        "skill_score": skill * 100,
        "experience_score": experience * 100,
        "skills": skills,
        "missing_skills": missing_skills
    })

In [ ]:
max_score = max([c["score"] for c in results])

for c in results:
    c["normalized_score"] = (c["score"] / max_score) * 100

ranked_results = sorted(results, key=lambda x: x["score"], reverse=True)

print("\n🏆 FINAL SMART TALENT RANKING:\n")

for i, candidate in enumerate(ranked_results, start=1):

    print(f"{i}. {candidate['name']}")
    print(f"   Final Score: {candidate['score']:.2f}%")
    print(f"   Normalized: {candidate['normalized_score']:.2f}%")
    print(f"   Semantic: {candidate['semantic']:.2f}%")
    print(f"   Skill: {candidate['skill_score']:.2f}%")
    print(f"   Experience: {candidate['experience_score']:.2f}%")
    print(f"   Skills: {candidate['skills']}")
    print(f"   Missing Skills: {candidate['missing_skills']}")
    print(f"   Confidence: {confidence_level(candidate['score'])}")
    print(f"   Justification: {generate_justification(candidate)}")
    print(f"   Recommendation: {'SELECT' if candidate['score'] > 50 else 'REJECT'}")

    print("-" * 50)


🏆 FINAL SMART TALENT RANKING:

1. perfect_resume.pdf
   Final Score: 66.95%
   Normalized: 100.00%
   Semantic: 53.90%
   Skill: 100.00%
   Experience: 50.00%
   Skills: ['excel', 'communication', 'management']
   Missing Skills: []
   Confidence: Medium
   Justification: Strong candidate with good semantic alignment (53.90%) and relevant skills ['excel', 'communication', 'management'].
   Recommendation: SELECT
--------------------------------------------------
2. resume-sample(5).pdf
   Final Score: 54.57%
   Normalized: 81.51%
   Semantic: 45.14%
   Skill: 100.00%
   Experience: 10.00%
   Skills: ['excel', 'communication', 'management']
   Missing Skills: []
   Confidence: Medium
   Justification: Candidate has relevant skills ['excel', 'communication'] but moderate semantic alignment (45.14%).
   Recommendation: SELECT
--------------------------------------------------
3. resume-sample(3).pdf
   Final Score: 52.66%
   Normalized: 78.66%
   Semantic: 41.32%
   Skill: 100.00%
   Exp